# ☕ Preprocesamiento + Modelado
### Proyecto Grupal — Clasificación de Calidad de Café

---

## 🗺️ Hoja de Ruta de este Notebook

| Bloque | Qué vamos a hacer |
|--------|-------------------|
| **1** | Cargar el dataset limpio del EDA |
| **2** | Limpieza final (eliminar columnas inútiles) |
| **3** | Imputación de valores nulos |
| **4** | Encoding de variables categóricas |
| **5** | Escalado de variables numéricas |
| **6** | Split Train / Test |
| **7** | Modelos Baseline (Logistic Regression) |
| **8** | Modelos Avanzados (Random Forest) |
| **9** | Comparación de modelos y selección del mejor |
| **10** | Análisis del modelo ganador (Matriz de confusión, ROC, Feature Importance) |
| **11** | Guardar el modelo entrenado |

---


> ⚠️ **Importante:** Usamos `01_EDA_Merge_Jonathan.ipynb` con el CSV limpio guardado.

---
## 📦 BLOQUE 0 — Importar librerías

Añadimos librerías nuevas respecto al EDA:
- `sklearn` → preprocesamiento, modelos, métricas
- `xgboost` → modelo de boosting avanzado
- `joblib` → guardar/cargar el modelo entrenado

In [5]:
# ─── Librerías base ───────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ─── Preprocesamiento ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# ─── Modelos ──────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


# ─── Métricas ─────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay
)

# ─── Guardar modelo ───────────────────────────────────────────────────────────
import joblib

# ─── Configuración visual ─────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print('✅ Librerías importadas correctamente')

✅ Librerías importadas correctamente


---
## 📂 BLOQUE 1 — Cargar el dataset del EDA

Cargamos el CSV `coffee_quality_eda_fusion_clean_Jonathan.csv`

In [7]:
# Cargamos el dataset limpio generado por el EDA
df = pd.read_csv('../data/processed/coffee_quality_eda_fusion_clean_Jonathan.csv')

print(f'✅ Dataset cargado')
print(f'📊 Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'\n🎯 Distribución del target:')
print(df['quality_label'].value_counts())
df.head()

✅ Dataset cargado
📊 Dimensiones: 1512 filas x 23 columnas

🎯 Distribución del target:
quality_label
Specialty       834
No Specialty    678
Name: count, dtype: int64


,Acidity,Aftertaste,Aroma,Balance,Body,Category One Defects,Category Two Defects,Clean Cup,Color,Country of Origin,...,Moisture Percentage,Processing Method,Quakers,Region,Sweetness,Uniformity,Variety,altitud_limpia,dataset_source,quality_label
0,8.75,8.67,8.67,8.42,8.50,0,0,10.0,Green,Ethiopia,...,0.12,Washed / Wet,0.0,guji-hambela,10.0,10.0,NaN,2075.0,2018_volpatto,Specialty
1,8.58,8.50,8.75,8.42,8.42,0,1,10.0,Green,Ethiopia,...,0.12,Washed / Wet,0.0,guji-hambela,10.0,10.0,Other,2075.0,2018_volpatto,Specialty
2,8.42,8.42,8.42,8.42,8.33,0,0,10.0,NaN,Guatemala,...,0.00,NaN,0.0,NaN,10.0,10.0,Bourbon,1700.0,2018_volpatto,Specialty
3,8.42,8.42,8.17,8.25,8.50,0,2,10.0,Green,Ethiopia,...,0.11,Natural / Dry,0.0,oromia,10.0,10.0,NaN,2000.0,2018_volpatto,Specialty
4,8.50,8.25,8.25,8.33,8.42,0,2,10.0,Green,Ethiopia,...,0.12,Washed / Wet,0.0,guji-hambela,10.0,10.0,Other,2075.0,2018_volpatto,Specialty


In [10]:
print (df.columns)

Index(['Acidity', 'Aftertaste', 'Aroma', 'Balance', 'Body',
       'Category One Defects', 'Category Two Defects', 'Clean Cup', 'Color',
       'Country of Origin', 'Flavor', 'Grading Date', 'Harvest Year',
       'Moisture Percentage', 'Processing Method', 'Quakers', 'Region',
       'Sweetness', 'Uniformity', 'Variety', 'altitud_limpia',
       'dataset_source', 'quality_label'],
      dtype='str')


---
## 🧹 BLOQUE 2 — Limpieza final

### ¿Qué eliminamos y por qué?

| Columna | Motivo de eliminación |
|---------|----------------------|
| `Uniformity` | Varianza casi 0 (todos valen ~10). El modelo no aprende nada de ella |
| `Sweetness` | Todos los valores son exactamente 10. Sin varianza |
| `Clean Cup` | Igual que Sweetness, sin variación |
| Metadata | Que aún tengamos. |

**Regla:** Una variable sin varianza es como un semáforo que siempre está en verde — no te da información sobre cuándo parar.

In [12]:
# ─── Columnas a eliminar ──────────────────────────────────────────────────────

# Metadata que no aporta valor predictivo
metadata = [
    'Unnamed: 0', 'ID', 'Farm Name', 'Lot Number', 'Mill',
    'ICO Number', 'Company', 'In-Country Partner', 'Producer',
    'Bag Weight', 'Number of Bags', 'Expiration',
    'Certification Body', 'Certification Address', 'Certification Contact',
    'Owner', 'Grading Date', 'Status', 'Region',
    'Harvest Year',       # Fecha de cosecha — no predice calidad
    'dataset_source'      # Columna de auditoría — solo indica origen del dato
]

# Variables sin varianza (inútiles para el modelo)
sin_varianza = ['Uniformity', 'Sweetness', 'Clean Cup', 'Defects']

# Unimos todas las columnas a eliminar
cols_eliminar = metadata + sin_varianza

# Solo eliminamos las que realmente existen en el dataset
cols_eliminar = [c for c in cols_eliminar if c in df.columns]

df_model = df.drop(columns=cols_eliminar)

print(f'✅ Columnas eliminadas: {len(cols_eliminar)}')
print(f'📊 Dimensiones restantes: {df_model.shape[0]} filas x {df_model.shape[1]} columnas')
print(f'\n📝 Columnas que quedan:')
for i, col in enumerate(df_model.columns, 1):
    print(f'  {i:02d}. {col} ({df_model[col].dtype})')

✅ Columnas eliminadas: 7
📊 Dimensiones restantes: 1512 filas x 16 columnas

📝 Columnas que quedan:
  01. Acidity (float64)
  02. Aftertaste (float64)
  03. Aroma (float64)
  04. Balance (float64)
  05. Body (float64)
  06. Category One Defects (int64)
  07. Category Two Defects (int64)
  08. Color (str)
  09. Country of Origin (str)
  10. Flavor (float64)
  11. Moisture Percentage (float64)
  12. Processing Method (str)
  13. Quakers (float64)
  14. Variety (str)
  15. altitud_limpia (float64)
  16. quality_label (str)


---
## 🩹 BLOQUE 3 — Imputación de valores nulos

### ¿Qué es imputar?
Rellenar los valores vacíos con un valor calculado para no perder esas filas.

### Estrategia:
- **Variables numéricas** → rellenamos con la **mediana** (más robusta que la media ante outliers)
- **Variables categóricas** → rellenamos con la **moda** (el valor más frecuente)

### ¿Por qué mediana y no media?
Si la altitud tiene un outlier de 5400m, la media se dispara pero la mediana no se mueve.

In [13]:
# ─── Ver nulos restantes ──────────────────────────────────────────────────────
nulos = df_model.isnull().sum()
print('🕳️ Nulos antes de imputar:')
print(nulos[nulos > 0])

# ─── Separar columnas por tipo ────────────────────────────────────────────────
# Excluimos el target de la imputación
cols_numericas = df_model.select_dtypes(include=[np.number]).columns.tolist()
cols_categoricas = df_model.select_dtypes(include=['object']).columns.tolist()
cols_categoricas = [c for c in cols_categoricas if c != 'quality_label']

print(f'\n🔢 Numéricas: {cols_numericas}')
print(f'📝 Categóricas: {cols_categoricas}')

# ─── Imputar numéricas con mediana ────────────────────────────────────────────
imp_num = SimpleImputer(strategy='median')
df_model[cols_numericas] = imp_num.fit_transform(df_model[cols_numericas])

# ─── Imputar categóricas con moda ─────────────────────────────────────────────
imp_cat = SimpleImputer(strategy='most_frequent')
df_model[cols_categoricas] = imp_cat.fit_transform(df_model[cols_categoricas])

print(f'\n✅ Nulos tras imputar: {df_model.isnull().sum().sum()}')

🕳️ Nulos antes de imputar:
Color                266
Country of Origin      1
Processing Method    156
Quakers                1
Variety              206
altitud_limpia       232
dtype: int64

🔢 Numéricas: ['Acidity', 'Aftertaste', 'Aroma', 'Balance', 'Body', 'Category One Defects', 'Category Two Defects', 'Flavor', 'Moisture Percentage', 'Quakers', 'altitud_limpia']
📝 Categóricas: ['Color', 'Country of Origin', 'Processing Method', 'Variety']

✅ Nulos tras imputar: 0


---
## 🔤 BLOQUE 4 — Encoding de variables categóricas

### ¿Por qué hay que encodear?
Los modelos de ML solo entienden números. El texto `'Specialty'` no significa nada para un algoritmo.
Tenemos que convertir cada categoría a un número.

### Métodos:
- **LabelEncoder** → Asigna un número entero a cada categoría (0, 1, 2...). Útil para árboles de decisión.
- **OneHotEncoder** → Crea una columna nueva por cada categoría con 0 o 1. Mejor para regresión logística.

Usamos **LabelEncoder** porque Random Forest y XGBoost lo manejan bien y no genera demasiadas columnas.

In [14]:
# ─── Encodear variables categóricas con LabelEncoder ─────────────────────────
# Guardamos los encoders para poder usarlos en la app de predicción
encoders = {}

for col in cols_categoricas:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    encoders[col] = le
    print(f'  ✅ {col}: {len(le.classes_)} categorías únicas')

# ─── Encodear el target ───────────────────────────────────────────────────────
# Specialty = 1, No Specialty = 0
df_model['quality_label'] = (df_model['quality_label'] == 'Specialty').astype(int)

print(f'\n✅ Encoding completado')
print(f'🎯 Target: Specialty=1, No Specialty=0')
print(df_model['quality_label'].value_counts())

  ✅ Color: 6 categorías únicas
  ✅ Country of Origin: 37 categorías únicas
  ✅ Processing Method: 12 categorías únicas
  ✅ Variety: 62 categorías únicas

✅ Encoding completado
🎯 Target: Specialty=1, No Specialty=0
quality_label
1    834
0    678
Name: count, dtype: int64


---
## 🔗 BLOQUE 4b — Análisis de Multicolinealidad (VIF)

**¿Qué es la multicolinealidad?**  
Ocurre cuando dos o más variables predictoras están muy correlacionadas entre sí.  
En la matriz de correlación del EDA vimos que Flavor–Aftertaste tienen r=0.90, Balance–Aftertaste r=0.83, etc.  

**¿Por qué es un problema?**  
- Para **Logistic Regression**: infla los coeficientes y los hace inestables
- Para **Random Forest / XGBoost**: no es un problema grave, los árboles lo manejan solos

**Métrica: VIF (Variance Inflation Factor)**  
- VIF < 5 → sin problema  
- VIF 5–10 → multicolinealidad moderada, vigilar  
- VIF > 10 → multicolinealidad severa, considerar eliminar o trabajar con Ride/Lasso

In [16]:
# ─── VIF para detectar multicolinealidad ──────────────────────────────────────
# Solo sobre variables numéricas (las sensoriales, que son las sospechosas)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Tomamos solo las columnas numéricas del dataset codificado
cols_num_vif = df_model.select_dtypes(include=[np.number]).drop(columns=['quality_label']).columns.tolist()
X_vif = df_model[cols_num_vif].copy()

vif_data = pd.DataFrame()
vif_data['Variable'] = cols_num_vif
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)

print('📊 Variance Inflation Factor (VIF) por variable:')
print(vif_data.to_string(index=False))
print()
print('📌 Referencia: VIF < 5 = OK | VIF 5-10 = Moderado | VIF > 10 = Severo')
print()
print('💡 Nota: Para Random Forest y XGBoost los VIF altos no son críticos.')
print('         Solo afecta a Logistic Regression — cuyos coeficientes son inestables si VIF > 10.')

📊 Variance Inflation Factor (VIF) por variable:
            Variable         VIF
              Flavor 2676.546440
          Aftertaste 2309.456278
             Balance 1469.310440
             Acidity 1466.763797
                Body 1366.546860
               Aroma 1308.674188
   Processing Method   13.096932
               Color    9.266247
   Country of Origin    3.326989
             Variety    3.282018
Category Two Defects    1.759779
 Moisture Percentage    1.310019
Category One Defects    1.217493
             Quakers    1.148224
      altitud_limpia    1.058297

📌 Referencia: VIF < 5 = OK | VIF 5-10 = Moderado | VIF > 10 = Severo

💡 Nota: Para Random Forest y XGBoost los VIF altos no son críticos.
         Solo afecta a Logistic Regression — cuyos coeficientes son inestables si VIF > 10.


---
## ⚖️ BLOQUE 5 — Escalado de variables numéricas

### ¿Por qué escalar?
Imagina que tienes Aroma (valores entre 6.5 y 8.5) y altitud (valores entre 500 y 2500).
Los modelos basados en distancias o gradientes (como Logistic Regression) se "dejan llevar"
por la variable con números más grandes. El escalado iguala el campo de juego.

### ¿Cuándo NO es necesario?
Los modelos basados en árboles (Decision Tree, Random Forest, XGBoost) **no necesitan escalado**.
Pero lo aplicamos igual para que Logistic Regression funcione correctamente.

### StandardScaler:
Transforma cada variable para que tenga **media = 0** y **desviación estándar = 1**.

In [17]:
# ─── Separar features y target ────────────────────────────────────────────────
X = df_model.drop(columns=['quality_label'])
y = df_model['quality_label']

print(f'Features (X): {X.shape}')
print(f'Target  (y): {y.shape}')
print(f'\nColumnas del modelo:')
print(list(X.columns))

Features (X): (1512, 15)
Target  (y): (1512,)

Columnas del modelo:
['Acidity', 'Aftertaste', 'Aroma', 'Balance', 'Body', 'Category One Defects', 'Category Two Defects', 'Color', 'Country of Origin', 'Flavor', 'Moisture Percentage', 'Processing Method', 'Quakers', 'Variety', 'altitud_limpia']


---
## ✂️ BLOQUE 6 — Split Train / Test

### ¿Por qué dividir el dataset?
El modelo aprende con los datos de entrenamiento (train). Los datos de test los guarda en un cajón y
solo los abre al final para ver si el modelo generaliza bien o si hizo trampa (memorización = overfitting).

### `stratify=y`
Garantiza que tanto train como test tengan la misma proporción de Specialty/No Specialty.
Sin esto, podría ocurrir que el test tenga muy pocos No Specialty y las métricas sean engañosas.

### División: 80% train / 20% test
Con 1512 filas: ~1209 para entrenar, ~303 para testear.

In [18]:
# ─── Split ────────────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% para test
    random_state=42,      # semilla para reproducibilidad (siempre el mismo split)
    stratify=y            # mantener proporciones de clases
)

print(f'📊 Train: {X_train.shape[0]} filas ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'📊 Test:  {X_test.shape[0]} filas ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\n🎯 Distribución en Train:')
print(y_train.value_counts())
print(f'\n🎯 Distribución en Test:')
print(y_test.value_counts())

# ─── Escalar DESPUÉS del split ────────────────────────────────────────────────
# MUY IMPORTANTE: el scaler aprende solo con train, no con test
# Si escalamos antes del split, habría data leakage (el test contamina el train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform en train
X_test_scaled  = scaler.transform(X_test)         # solo transform en test

print('\n✅ Escalado completado (fit solo en train, para evitar data leakage)')

📊 Train: 1209 filas (80.0%)
📊 Test:  303 filas (20.0%)

🎯 Distribución en Train:
quality_label
1    667
0    542
Name: count, dtype: int64

🎯 Distribución en Test:
quality_label
1    167
0    136
Name: count, dtype: int64

✅ Escalado completado (fit solo en train, para evitar data leakage)


---
## 🤖 BLOQUE 7 — Modelos Baseline

Empezamos con modelos simples. Sirven como **punto de referencia**:
si los modelos avanzados no mejoran al baseline, algo está mal.

### Métricas que usamos:

| Métrica | Qué mide | Cuándo es importante |
|---------|----------|----------------------|
| **Accuracy** | % de predicciones correctas | Con clases balanceadas |
| **Precision** | De los que predije Specialty, ¿cuántos realmente lo son? | Cuando el falso positivo es costoso |
| **Recall** | De todos los Specialty reales, ¿cuántos encontré? | Cuando el falso negativo es costoso |
| **F1** | Media armónica de Precision y Recall | Con clases desbalanceadas (**el más importante aquí**) |
| **ROC-AUC** | Capacidad de separar clases a todos los umbrales | Evaluación general del modelo |

In [19]:
# ─── Función para evaluar cualquier modelo ────────────────────────────────────
def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te, cv=5):
    """
    Entrena el modelo, predice en test y devuelve un diccionario con todas las métricas.
    También hace validación cruzada para ver si el modelo es estable.
    """
    # Entrenar
    modelo.fit(X_tr, y_tr)
    
    # Predecir en test
    y_pred = modelo.predict(X_te)
    y_prob = modelo.predict_proba(X_te)[:, 1]  # probabilidad de la clase positiva (Specialty)
    
    # Validación cruzada con F1 (más fiable con dataset pequeño y desbalanceado)
    cv_scores = cross_val_score(
        modelo, X_tr, y_tr,
        cv=StratifiedKFold(n_splits=cv, shuffle=True, random_state=42),
        scoring='f1'
    )
    
    return {
        'Modelo'     : nombre,
        'Accuracy'   : accuracy_score(y_te, y_pred),
        'Precision'  : precision_score(y_te, y_pred),
        'Recall'     : recall_score(y_te, y_pred),
        'F1'         : f1_score(y_te, y_pred),
        'ROC-AUC'    : roc_auc_score(y_te, y_prob),
        'CV F1 Media': cv_scores.mean(),
        'CV F1 Std'  : cv_scores.std(),
        'modelo_obj' : modelo
    }

print('✅ Función de evaluación lista')

✅ Función de evaluación lista


In [20]:
# ─── BASELINE 1: Logistic Regression ─────────────────────────────────────────
# Modelo lineal. Encuentra una "línea" que separa las dos clases.
# class_weight='balanced' compensa el desbalance 55/45 automáticamente.
# Necesita datos escalados (usamos X_train_scaled)
lr_ridge = LogisticRegression(penalty='l2', C=0.1, class_weight='balanced', max_iter=1000)

res_lr = evaluar_modelo('Logistic Regression', lr_ridge,
                        X_train_scaled, X_test_scaled, y_train, y_test)

print('📊 Logistic Regression:')
for k, v in res_lr.items():
    if k not in ['Modelo', 'modelo_obj']:
        print(f'  {k}: {v:.4f}')

📊 Logistic Regression:
  Accuracy: 0.9340
  Precision: 0.9401
  Recall: 0.9401
  F1: 0.9401
  ROC-AUC: 0.9738
  CV F1 Media: 0.9258
  CV F1 Std: 0.0125


---
## 💾 BLOQUE 8 — Guardar el modelo

Guardamos el modelo entrenado para usarlo en la app de Streamlit.
También guardamos el scaler y los encoders para poder transformar datos nuevos correctamente.

In [ ]:
# ─── Guardar modelo, scaler y encoders ───────────────────────────────────────
import os
os.makedirs('../models', exist_ok=True)

# Guardar el modelo ganador
joblib.dump(lr_ridge, '../models/model.pkl')